<a href="https://colab.research.google.com/github/D-F-V/Project_brief/blob/main/project_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo de apoyo a la valoración residencial en Madrid

## 0. Resumen ejecutivo

El proyecto estima el precio anunciado por m² para apoyar la revisión de anuncios de viviendas de Madrid de 2018.
Se seleccionó Random Forest tras compararlo con Ridge y dos referencias básicas, y realizar un ajuste temporal.
En el cuarto trimestre obtuvo un MAE de 516,87 €/m², un 52,01 % menor que la regla de comparables.
Se recomienda utilizarlo para orientar revisiones humanas, sin fijar precios automáticamente.
Sus principales límites son la antigüedad de los datos, el mayor error cerca del centro y que predice precios anunciados, no precios de cierre.

## 1. Del brief a una tarea de aprendizaje automático

El proyecto busca ayudar al responsable de valoraciones de una
agencia inmobiliaria ficticia a identificar anuncios cuyo precio
por metro cuadrado merece revisión frente a viviendas comparables.

Se utilizarán anuncios de Madrid correspondientes a 2018.
Es un ejercicio histórico: no estima precios actuales ni precios
finales de compraventa.

| Elemento | Definición |
|---|---|
| Decisión | Priorizar anuncios para revisar su precio de salida. |
| Tipo de tarea | Regresión: predecir un valor numérico. |
| Variable objetivo | UNITPRICE, precio anunciado en euros por metro cuadrado. |
| Unidad de predicción | Un registro de anuncio de vivienda. Los identificadores repetidos requieren tratamiento explícito. |
| Momento de predicción | Cuando el responsable revisa un anuncio y dispone de sus características. |
| Horizonte | Estimación para esa revisión, no una previsión de la venta futura. |
| Información candidata | Superficie, dormitorios, baños, ascensor, terraza, localización y otras características documentadas. |
| Variables excluidas como predictores | UNITPRICE, porque es la respuesta; PRICE, porque junto con la superficie permite reconstruirla; ASSETID, porque es un identificador. |
| Regla de trabajo propuesta | Comparar con la mediana de viviendas similares; es una regla del caso ficticio, no una práctica empresarial observada. |
| Referencia sencilla | Predecir siempre la mediana de UNITPRICE del entrenamiento. |
| Métrica principal | MAE, expresado en €/m². Se justificará según el uso y las consecuencias del error. |
| Conexión con el negocio | Utilizar la diferencia entre precio anunciado y referencia estimada para priorizar revisiones humanas. |

### Cambios respecto al primer brief

Se mantiene el dataset, el objetivo y el responsable de la decisión.
Las decisiones definitivas de limpieza y selección de variables
se documentarán en los siguientes apartados.

### Qué significa para el responsable de valoraciones

La predicción ofrecerá una referencia para revisar anuncios.
Una diferencia grande será un motivo para investigar, no una
prueba de que el precio sea incorrecto.

## 2. Datos y calidad

Se carga el CSV de Madrid_Sale utilizado en la primera entrega.
Contiene registros de anuncios de viviendas de Madrid durante
los cuatro trimestres de 2018.

La fuente de los datos y su diccionario es idealista18
(Rey-Blanco et al., s. f.).

Esta tabla conserva los datos anteriores a la limpieza definitiva.
Se comprobarán sus dimensiones, columnas y periodos antes de
aplicar cualquier transformación.

In [30]:
from time import perf_counter

inicio_notebook = perf_counter()

In [31]:
import pandas as pd
import numpy as np

# Semilla que utilizaremos en los pasos que incluyan aleatoriedad
SEED = 42

# CSV guardado en tu repositorio
URL_DATOS = (
    "https://raw.githubusercontent.com/D-F-V/Project_brief/"
    "main/project/data/Madrid_Sale_original.csv"
)

# Conservamos esta tabla como copia original
datos_originales = pd.read_csv(
    URL_DATOS,
    dtype={"ASSETID": "string"}
)

# Comprobaciones iniciales
assert datos_originales.shape == (94815, 41), (
    "Las dimensiones no coinciden con las de la primera entrega."
)

columnas_necesarias = {
    "ASSETID", "PERIOD", "PRICE",
    "UNITPRICE", "CONSTRUCTEDAREA"
}
assert columnas_necesarias.issubset(datos_originales.columns), (
    "Faltan columnas necesarias."
)

print("Filas:", datos_originales.shape[0])
print("Columnas:", datos_originales.shape[1])
print("Periodos:", sorted(datos_originales["PERIOD"].dropna().unique()))
print("Identificadores distintos:", datos_originales["ASSETID"].nunique())

display(datos_originales.head())

Filas: 94815
Columnas: 41
Periodos: [np.int64(201803), np.int64(201806), np.int64(201809), np.int64(201812)]
Identificadores distintos: 75804


,ASSETID,PERIOD,PRICE,UNITPRICE,CONSTRUCTEDAREA,ROOMNUMBER,BATHNUMBER,HASTERRACE,HASLIFT,HASAIRCONDITIONING,...,CADDWELLINGCOUNT,CADASTRALQUALITYID,BUILTTYPEID_1,BUILTTYPEID_2,BUILTTYPEID_3,DISTANCE_TO_CITY_CENTER,DISTANCE_TO_METRO,DISTANCE_TO_CASTELLANA,LONGITUDE,LATITUDE
0,A15019136831406238029,201803,126000.0,2680.851064,47,1,1,0,1,1,...,319,3.0,0,1,0,8.058429,0.872075,6.868677,-3.766933,40.362485
1,A6677225905472065344,201803,235000.0,4351.851852,54,1,1,0,0,0,...,11,3.0,0,0,1,0.876369,0.116382,1.544125,-3.710725,40.422430
2,A13341979748618524775,201803,373000.0,4973.333333,75,2,1,0,0,1,...,26,3.0,0,0,1,0.907479,0.139109,1.608444,-3.711571,40.422190
3,A4775182175615276542,201803,284000.0,5916.666667,48,1,1,0,1,1,...,15,5.0,0,0,1,0.845462,0.144299,1.516166,-3.710440,40.422251
4,A2492087730711701973,201803,228000.0,4560.000000,50,0,1,0,0,0,...,19,7.0,0,0,1,1.250231,0.337098,1.794136,-3.714340,40.408741


### 2.1. Comprobaciones iniciales de calidad

Se cuentan los valores ausentes, los duplicados exactos y las
repeticiones del identificador. Un identificador repetido no
implica que las filas sean copias ni que deban eliminarse.

En este paso no se modifica ningún dato. El tratamiento de los
problemas encontrados se justificará después.

In [32]:
# 1. Valores ausentes
nulos = datos_originales.isna().sum()

tabla_nulos = pd.DataFrame({
    "Valores ausentes": nulos,
    "Porcentaje": (nulos / len(datos_originales) * 100).round(2)
})

print("COLUMNAS CON VALORES AUSENTES")
display(
    tabla_nulos[tabla_nulos["Valores ausentes"] > 0]
    .sort_values("Valores ausentes", ascending=False)
)

# 2. Duplicados exactos e identificadores repetidos
frecuencia_id = datos_originales["ASSETID"].value_counts()

print(
    "Duplicados exactos:",
    datos_originales.duplicated().sum()
)
print(
    "Identificadores que aparecen más de una vez:",
    (frecuencia_id > 1).sum()
)
print(
    "Filas correspondientes a esos identificadores:",
    frecuencia_id[frecuencia_id > 1].sum()
)

# 3. Comprobar si un identificador aparece en varios trimestres
periodos_por_id = (
    datos_originales.groupby("ASSETID")["PERIOD"].nunique()
)

print(
    "Identificadores presentes en más de un trimestre:",
    (periodos_por_id > 1).sum()
)

# 4. Tamaño de cada periodo
print("\nREGISTROS E IDENTIFICADORES POR PERIODO")
display(
    datos_originales.groupby("PERIOD").agg(
        registros=("ASSETID", "size"),
        identificadores_distintos=("ASSETID", "nunique")
    )
)

COLUMNAS CON VALORES AUSENTES


,Valores ausentes,Porcentaje
CONSTRUCTIONYEAR,55873,58.93
FLATLOCATIONID,6387,6.74
FLOORCLEAN,3846,4.06
CADASTRALQUALITYID,1,0.00


Duplicados exactos: 0
Identificadores que aparecen más de una vez: 13829
Filas correspondientes a esos identificadores: 32840
Identificadores presentes en más de un trimestre: 0

REGISTROS E IDENTIFICADORES POR PERIODO


,registros,identificadores_distintos
PERIOD,,
201803,21920,17622
201806,12652,12518
201809,15973,14246
201812,44270,31418


### 2.2. Limpieza inicial y decisiones

Se conserva una copia original. Solo se excluyen registros sin
superficie o precio por metro cuadrado válidos para esta tarea.

Las anomalías de dormitorios y localización se marcan y sus
valores afectados se convierten en ausentes, sin eliminar el
anuncio completo. El criterio de menos de 5 m² por dormitorio
es una regla de revisión adoptada para este proyecto, no una
norma general de vivienda.

Las viviendas grandes con pocos dormitorios y las que declaran
cero baños se conservan, porque no se ha demostrado que sean errores.
Los identificadores repetidos tampoco se eliminan automáticamente.

La imputación de valores ausentes se realizará posteriormente,
ajustada únicamente con los datos de entrenamiento.

In [33]:
datos_limpios = datos_originales.copy()

# 1. Excluir solo registros sin superficie u objetivo válidos
columnas_esenciales = ["CONSTRUCTEDAREA", "UNITPRICE"]

valores_validos = (
    np.isfinite(datos_limpios[columnas_esenciales]).all(axis=1)
    & datos_limpios[columnas_esenciales].gt(0).all(axis=1)
)

filas_excluidas = int((~valores_validos).sum())
datos_limpios = datos_limpios.loc[valores_validos].copy()

# 2. Marcar dormitorios extremadamente poco plausibles
# Evitamos dividir por cero: cero dormitorios puede indicar un estudio
dormitorios = datos_limpios["ROOMNUMBER"]
m2_por_dormitorio = (
    datos_limpios["CONSTRUCTEDAREA"]
    / dormitorios.where(dormitorios > 0)
)

datos_limpios["flag_dormitorios_dudosos"] = (
    (dormitorios > 0) & (m2_por_dormitorio < 5)
)

datos_limpios.loc[
    datos_limpios["flag_dormitorios_dudosos"],
    "ROOMNUMBER"
] = np.nan

# 3. Marcar el registro geográfico anómalo ya inspeccionado
# Aplicamos la decisión al caso concreto, no a toda vivienda lejana
datos_limpios["flag_geografia_dudosa"] = (
    datos_limpios["ASSETID"].eq("A7501251371917084834")
    & datos_limpios["PERIOD"].eq(201806)
)

columnas_geograficas = [
    "LONGITUDE",
    "LATITUDE",
    "DISTANCE_TO_CITY_CENTER",
    "DISTANCE_TO_METRO",
    "DISTANCE_TO_CASTELLANA"
]

datos_limpios.loc[
    datos_limpios["flag_geografia_dudosa"],
    columnas_geograficas
] = np.nan

# 4. Registro de cambios
resumen_limpieza = pd.DataFrame({
    "Comprobación": [
        "Filas originales",
        "Filas excluidas por superficie u objetivo inválidos",
        "Filas conservadas",
        "Registros con dormitorios convertidos en ausentes",
        "Registros con geografía convertida en ausente"
    ],
    "Cantidad": [
        len(datos_originales),
        filas_excluidas,
        len(datos_limpios),
        int(datos_limpios["flag_dormitorios_dudosos"].sum()),
        int(datos_limpios["flag_geografia_dudosa"].sum())
    ]
})

display(resumen_limpieza)

,Comprobación,Cantidad
0,Filas originales,94815
1,Filas excluidas por superficie u objetivo invá...,0
2,Filas conservadas,94815
3,Registros con dormitorios convertidos en ausentes,2
4,Registros con geografía convertida en ausente,1


### Qué significa para el responsable de valoraciones

La limpieza inicial conserva los 94.815 registros. Se han marcado
dos anuncios con un número de dormitorios muy poco plausible y
uno con localización incoherente, dejando esos valores como ausentes.

Esto permite conservar el resto de la información sin utilizar
directamente los valores sospechosos. Todavía quedan valores
ausentes y registros repetidos cuyo tratamiento se completará
durante la preparación y evaluación del modelo.

In [34]:
# Separación temporal: aprender del pasado y evaluar en periodos posteriores
entrenamiento = datos_limpios[
    datos_limpios["PERIOD"].isin([201803, 201806])
].copy()

validacion = datos_limpios[
    datos_limpios["PERIOD"].eq(201809)
].copy()

prueba = datos_limpios[
    datos_limpios["PERIOD"].eq(201812)
].copy()

# Comprobar que ningún identificador aparece en dos conjuntos
ids_train = set(entrenamiento["ASSETID"])
ids_val = set(validacion["ASSETID"])
ids_test = set(prueba["ASSETID"])

assert ids_train.isdisjoint(ids_val)
assert ids_train.isdisjoint(ids_test)
assert ids_val.isdisjoint(ids_test)

assert (
    len(entrenamiento) + len(validacion) + len(prueba)
    == len(datos_limpios)
)

resumen_division = pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Validación", "Prueba final"],
    "Filas": [
        len(entrenamiento),
        len(validacion),
        len(prueba)
    ],
    "Identificadores distintos": [
        entrenamiento["ASSETID"].nunique(),
        validacion["ASSETID"].nunique(),
        prueba["ASSETID"].nunique()
    ]
})

display(resumen_division)
print("Comprobado: ningún ASSETID se comparte entre conjuntos.")

,Conjunto,Filas,Identificadores distintos
0,Entrenamiento,34572,30140
1,Validación,15973,14246
2,Prueba final,44270,31418


Comprobado: ningún ASSETID se comparte entre conjuntos.


## 3. Qué muestra la exploración

La exploración de esta fase utiliza únicamente los datos de
entrenamiento, correspondientes a los dos primeros trimestres de
2018. Validación y prueba quedan reservadas para evaluar modelos.

Se estudia primero la distribución de UNITPRICE. Después se
contrastarán las hipótesis del brief sobre distancia, superficie,
ascensor y terraza.

Los gráficos cuentan registros de anuncios, no necesariamente
viviendas distintas. Las repeticiones de ASSETID pueden dar más
peso a algunos inmuebles.

In [35]:
import plotly.express as px

# Resumen numérico del precio por metro cuadrado
resumen_precio = (
    entrenamiento["UNITPRICE"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.95])
    .round(2)
    .rename("Precio anunciado (€/m²)")
)

display(resumen_precio.to_frame())

# Distribución del objetivo: solo entrenamiento
fig = px.histogram(
    entrenamiento,
    x="UNITPRICE",
    nbins=50,
    title="Distribución del precio anunciado por m² — entrenamiento",
    labels={"UNITPRICE": "Precio anunciado (€/m²)"},
    template="plotly_white"
)

fig.update_layout(
    yaxis_title="Número de registros",
    bargap=0.05,
    showlegend=False
)

fig.add_vline(
    x=entrenamiento["UNITPRICE"].median(),
    line_dash="dash",
    line_color="red",
    annotation_text="Mediana",
    annotation_position="top right"
)

fig.show()

,Precio anunciado (€/m²)
count,34572.00
mean,3537.93
std,1686.13
min,805.31
25%,2122.81
50%,3338.00
75%,4610.22
95%,6602.88
max,9997.56


### Distribución del precio anunciado por m²

En entrenamiento, la mediana de UNITPRICE es 3.338 €/m² y la
media es 3.537,93 €/m². La distribución presenta una cola hacia
precios elevados. El 50 % central de los registros está entre
2.122,81 y 4.610,22 €/m².

Esta variación justifica analizar las características de las
viviendas antes de utilizar una única referencia de precio.
Los precios altos no se consideran errores por sí solos.

In [36]:
# Copia de entrenamiento para explorar H1
exploracion_h1 = entrenamiento.dropna(
    subset=["DISTANCE_TO_CITY_CENTER", "UNITPRICE"]
).copy()

# Tramos fijos de distancia en kilómetros
exploracion_h1["tramo_distancia"] = pd.cut(
    exploracion_h1["DISTANCE_TO_CITY_CENTER"],
    bins=[0, 2, 4, 6, 8, np.inf],
    labels=["0–2 km", "2–4 km", "4–6 km", "6–8 km", "Más de 8 km"],
    include_lowest=True
)

tabla_h1 = (
    exploracion_h1.groupby("tramo_distancia", observed=True)
    .agg(
        registros=("UNITPRICE", "size"),
        inmuebles_distintos=("ASSETID", "nunique"),
        mediana_eur_m2=("UNITPRICE", "median")
    )
)

display(tabla_h1.round(2))

# Asociación entre distancia y precio por m²
spearman_h1 = exploracion_h1["DISTANCE_TO_CITY_CENTER"].corr(
    exploracion_h1["UNITPRICE"],
    method="spearman"
)

print(f"Correlación de Spearman: {spearman_h1:.3f}")

# Gráfico de medianas por distancia
fig_h1 = px.bar(
    tabla_h1.reset_index(),
    x="tramo_distancia",
    y="mediana_eur_m2",
    title="Precio mediano por distancia al centro — entrenamiento",
    labels={
        "tramo_distancia": "Distancia al centro (km)",
        "mediana_eur_m2": "Precio anunciado mediano (€/m²)"
    },
    template="plotly_white"
)

fig_h1.show()

,registros,inmuebles_distintos,mediana_eur_m2
tramo_distancia,,,
0–2 km,6817,5898,4703.70
2–4 km,9795,8693,4073.53
4–6 km,8487,7433,2386.67
6–8 km,5373,4660,2350.65
Más de 8 km,4099,3526,2600.00


Correlación de Spearman: -0.532


### H1. Distancia al centro y precio por m²

En entrenamiento, la correlación de Spearman entre distancia
al centro y UNITPRICE es −0,532. La mediana disminuye de
4.703,70 €/m² en el tramo de 0–2 km a 2.350,65 €/m² en
6–8 km, pero aumenta a 2.600 €/m² a más de 8 km.

Los resultados apoyan inicialmente una asociación negativa,
aunque no uniforme. Falta comprobar si se mantiene al comparar
viviendas de tamaños similares. No se demuestra causalidad.

In [37]:
# Grupos de superficie para comparar viviendas de tamaño parecido
exploracion_h1["tramo_superficie"] = pd.cut(
    exploracion_h1["CONSTRUCTEDAREA"],
    bins=[0, 60, 90, 120, np.inf],
    labels=["Hasta 60 m²", "Más de 60–90 m²",
            "Más de 90–120 m²", "Más de 120 m²"],
    include_lowest=True
)

resultados_h1 = []

for tramo, grupo in exploracion_h1.groupby(
    "tramo_superficie", observed=True
):
    resultados_h1.append({
        "Superficie": tramo,
        "Registros": len(grupo),
        "Inmuebles distintos": grupo["ASSETID"].nunique(),
        "Spearman distancia-precio": (
            grupo["DISTANCE_TO_CITY_CENTER"]
            .corr(grupo["UNITPRICE"], method="spearman")
        )
    })

tabla_h1_superficie = pd.DataFrame(resultados_h1)
display(tabla_h1_superficie.round(3))

,Superficie,Registros,Inmuebles distintos,Spearman distancia-precio
0,Hasta 60 m²,8021,7144,-0.610
1,Más de 60–90 m²,11658,10362,-0.438
2,Más de 90–120 m²,6892,5943,-0.478
3,Más de 120 m²,8000,6756,-0.431


Al comparar por superficie, la correlación entre distancia y precio
por m² sigue siendo negativa en los cuatro grupos: −0,610, −0,438,
−0,478 y −0,431, respectivamente.

H1 queda apoyada descriptivamente en entrenamiento. La distancia
será una variable candidata y un criterio para seleccionar
comparables. Esta agrupación reduce las diferencias de tamaño,
pero no controla todas las características ni elimina el efecto
de los anuncios repetidos.

In [38]:
# H2: usamos entrenamiento y conservamos los tramos ya definidos
exploracion_h2 = exploracion_h1.copy()

print(
    "Spearman superficie-precio, con geografía disponible:",
    round(
        exploracion_h2["CONSTRUCTEDAREA"].corr(
            exploracion_h2["UNITPRICE"],
            method="spearman"
        ),
        3
    )
)

# Precio por grupo de superficie
tabla_h2 = (
    exploracion_h2.groupby("tramo_superficie", observed=True)
    .agg(
        registros=("UNITPRICE", "size"),
        inmuebles_distintos=("ASSETID", "nunique"),
        mediana_eur_m2=("UNITPRICE", "median")
    )
)

display(tabla_h2.round(2))

# Comprobar la relación dentro de cada tramo de distancia
resultados_h2 = []

for tramo, grupo in exploracion_h2.groupby(
    "tramo_distancia", observed=True
):
    resultados_h2.append({
        "Distancia": tramo,
        "Registros": len(grupo),
        "Inmuebles distintos": grupo["ASSETID"].nunique(),
        "Spearman superficie-precio": (
            grupo["CONSTRUCTEDAREA"]
            .corr(grupo["UNITPRICE"], method="spearman")
        )
    })

display(pd.DataFrame(resultados_h2).round(3))

# Gráfico de distribución por superficie
fig_h2 = px.box(
    exploracion_h2,
    x="tramo_superficie",
    y="UNITPRICE",
    points=False,
    title="Precio por m² según superficie — entrenamiento",
    labels={
        "tramo_superficie": "Superficie construida (m²)",
        "UNITPRICE": "Precio anunciado (€/m²)"
    },
    template="plotly_white"
)
fig_h2.update_xaxes(
    categoryorder="array",
    categoryarray=[
        "Hasta 60 m²",
        "Más de 60–90 m²",
        "Más de 90–120 m²",
        "Más de 120 m²"
    ]
)

fig_h2.show()

Spearman superficie-precio, con geografía disponible: 0.142


,registros,inmuebles_distintos,mediana_eur_m2
tramo_superficie,,,
Hasta 60 m²,8021,7144,3619.05
Más de 60–90 m²,11658,10362,2442.86
Más de 90–120 m²,6892,5943,3160.83
Más de 120 m²,8000,6756,4177.25


,Distancia,Registros,Inmuebles distintos,Spearman superficie-precio
0,0–2 km,6817,5898,-0.033
1,2–4 km,9795,8693,0.183
2,4–6 km,8487,7433,0.143
3,6–8 km,5373,4660,0.318
4,Más de 8 km,4099,3526,0.252


### H2. Superficie y precio por m²

H2 queda refutada como tendencia general en el entrenamiento
analizado. Spearman es +0,142 y las medianas no disminuyen
progresivamente con la superficie. Por distancia, la asociación
es prácticamente nula en 0–2 km y positiva en los demás tramos.

No se aplicará una reducción automática del precio por m² por
tener mayor superficie. Se conservará la superficie como variable
candidata, porque puede aportar información mediante relaciones
no lineales y combinaciones con otras características.

In [39]:
# Solo entrenamiento; excluimos plantas ausentes de esta comparación
exploracion_h3 = entrenamiento.dropna(
    subset=["FLOORCLEAN", "HASLIFT", "UNITPRICE"]
).copy()

# Nos centramos en plantas 0–6.
# Evitamos interpretar aquí los códigos extremos y los grupos
# de plantas altas con muy pocos anuncios sin ascensor.
exploracion_h3 = exploracion_h3[
    exploracion_h3["FLOORCLEAN"].between(0, 6)
].copy()

exploracion_h3["tramo_planta"] = pd.cut(
    exploracion_h3["FLOORCLEAN"],
    bins=[-1, 0, 2, 4, 6],
    labels=["Planta 0", "Plantas 1–2", "Plantas 3–4", "Plantas 5–6"]
)

tabla_h3 = (
    exploracion_h3.groupby(
        ["tramo_planta", "HASLIFT"],
        observed=True
    )
    .agg(
        registros=("UNITPRICE", "size"),
        inmuebles_distintos=("ASSETID", "nunique"),
        mediana_eur_m2=("UNITPRICE", "median")
    )
    .reset_index()
)

tabla_h3["Ascensor"] = tabla_h3["HASLIFT"].map({
    0: "Sin ascensor",
    1: "Con ascensor"
})

display(
    tabla_h3[
        ["tramo_planta", "Ascensor", "registros",
         "inmuebles_distintos", "mediana_eur_m2"]
    ].round(2)
)

,tramo_planta,Ascensor,registros,inmuebles_distintos,mediana_eur_m2
0,Planta 0,Sin ascensor,1822,1632,2264.91
1,Planta 0,Con ascensor,1844,1562,3142.14
2,Plantas 1–2,Sin ascensor,4716,4216,2120.61
3,Plantas 1–2,Con ascensor,8996,7795,3712.89
4,Plantas 3–4,Sin ascensor,3260,2845,2087.98
5,Plantas 3–4,Con ascensor,6397,5604,3870.59
6,Plantas 5–6,Sin ascensor,292,255,1849.39
7,Plantas 5–6,Con ascensor,3269,2837,4039.22


In [40]:
# Añadir grupos de superficie y distancia a la exploración de H3
h3_comparables = exploracion_h3.dropna(
    subset=["CONSTRUCTEDAREA", "DISTANCE_TO_CITY_CENTER"]
).copy()

h3_comparables["tramo_superficie"] = pd.cut(
    h3_comparables["CONSTRUCTEDAREA"],
    bins=[0, 60, 90, 120, np.inf],
    labels=["Hasta 60", "60–90", "90–120", "Más de 120"],
    include_lowest=True
)

h3_comparables["tramo_distancia"] = pd.cut(
    h3_comparables["DISTANCE_TO_CITY_CENTER"],
    bins=[0, 2, 4, 6, 8, np.inf],
    labels=["0–2", "2–4", "4–6", "6–8", "Más de 8"],
    include_lowest=True
)

grupos_h3 = [
    "tramo_planta", "tramo_superficie", "tramo_distancia"
]

resumen_h3 = (
    h3_comparables.groupby(
        grupos_h3 + ["HASLIFT"], observed=True
    )
    .agg(
        mediana=("UNITPRICE", "median"),
        inmuebles=("ASSETID", "nunique")
    )
    .reset_index()
)

sin_ascensor = (
    resumen_h3[resumen_h3["HASLIFT"] == 0]
    .drop(columns="HASLIFT")
)

con_ascensor = (
    resumen_h3[resumen_h3["HASLIFT"] == 1]
    .drop(columns="HASLIFT")
)

comparacion_h3 = sin_ascensor.merge(
    con_ascensor,
    on=grupos_h3,
    suffixes=("_sin", "_con")
)

comparacion_h3 = comparacion_h3[
    (comparacion_h3["inmuebles_sin"] >= 30)
    & (comparacion_h3["inmuebles_con"] >= 30)
].copy()

comparacion_h3["diferencia_eur_m2"] = (
    comparacion_h3["mediana_con"]
    - comparacion_h3["mediana_sin"]
)

print("Grupos comparables con suficiente cobertura:", len(comparacion_h3))

if not comparacion_h3.empty:
    print(
        "Grupos con mayor mediana con ascensor:",
        int((comparacion_h3["diferencia_eur_m2"] > 0).sum())
    )

    resumen_por_planta = (
        comparacion_h3.groupby("tramo_planta", observed=True)
        .agg(
            grupos_comparables=("diferencia_eur_m2", "size"),
            grupos_diferencia_positiva=(
                "diferencia_eur_m2", lambda s: (s > 0).sum()
            ),
            mediana_diferencias_eur_m2=("diferencia_eur_m2", "median")
        )
    )

    display(resumen_por_planta.round(2))
else:
    print("No hay grupos que cumplan el mínimo en ambos casos.")

Grupos comparables con suficiente cobertura: 46
Grupos con mayor mediana con ascensor: 45


,grupos_comparables,grupos_diferencia_positiva,mediana_diferencias_eur_m2
tramo_planta,,,
Planta 0,10,10,568.55
Plantas 1–2,18,17,1006.93
Plantas 3–4,15,15,1155.89
Plantas 5–6,3,3,1032.66


### H3. Ascensor, planta y precio por m²

Al comparar anuncios del mismo tramo de planta, superficie y
distancia, 45 de los 46 grupos con al menos 30 identificadores
distintos por categoría presentan mayor mediana de precio con
ascensor.

La mediana de las diferencias entre grupos es de 568,55 €/m²
en planta 0, 1.006,93 en plantas 1–2, 1.155,89 en plantas 3–4
y 1.032,66 en plantas 5–6. Este último tramo solo incluye tres
grupos comparables.

H3 queda apoyada parcialmente: la asociación positiva con ascensor
es consistente, pero no se observa un aumento continuo de la
diferencia con la planta. Se conservarán ascensor y planta como
variables candidatas, sin aplicar una prima fija ni atribuir
causalidad. Persisten diferencias no controladas y anuncios repetidos.

In [41]:
# Reutilizamos los tramos de superficie y distancia ya creados.
# exploracion_h1 contiene únicamente datos de entrenamiento.
h4 = exploracion_h1.dropna(subset=["HASTERRACE"]).copy()

grupos_h4 = ["tramo_superficie", "tramo_distancia"]

resumen_h4 = (
    h4.groupby(grupos_h4 + ["HASTERRACE"], observed=True)
    .agg(
        mediana=("UNITPRICE", "median"),
        inmuebles=("ASSETID", "nunique")
    )
    .reset_index()
)

sin_terraza = (
    resumen_h4[resumen_h4["HASTERRACE"] == 0]
    .drop(columns="HASTERRACE")
)

con_terraza = (
    resumen_h4[resumen_h4["HASTERRACE"] == 1]
    .drop(columns="HASTERRACE")
)

comparacion_h4 = sin_terraza.merge(
    con_terraza,
    on=grupos_h4,
    suffixes=("_sin", "_con")
)

comparacion_h4 = comparacion_h4[
    (comparacion_h4["inmuebles_sin"] >= 30)
    & (comparacion_h4["inmuebles_con"] >= 30)
].copy()

comparacion_h4["diferencia_eur_m2"] = (
    comparacion_h4["mediana_con"]
    - comparacion_h4["mediana_sin"]
)

print("Grupos comparables:", len(comparacion_h4))
print(
    "Grupos con mayor mediana con terraza:",
    int((comparacion_h4["diferencia_eur_m2"] > 0).sum())
)

display(comparacion_h4.round(2))

if not comparacion_h4.empty:
    comparacion_h4["grupo"] = (
        comparacion_h4["tramo_superficie"].astype(str)
        + " | "
        + comparacion_h4["tramo_distancia"].astype(str)
    )

    fig_h4 = px.bar(
        comparacion_h4,
        x="grupo",
        y="diferencia_eur_m2",
        title="Diferencia de precio con terraza frente a sin terraza",
        labels={
            "grupo": "Grupo de superficie y distancia al centro",
            "diferencia_eur_m2": "Diferencia de medianas (€/m²)"
        },
        template="plotly_white"
    )

    fig_h4.add_hline(y=0, line_color="black")
    fig_h4.update_xaxes(tickangle=-60)
    fig_h4.show()

Grupos comparables: 20
Grupos con mayor mediana con terraza: 5


,tramo_superficie,tramo_distancia,mediana_sin,inmuebles_sin,mediana_con,inmuebles_con,diferencia_eur_m2
0,Hasta 60 m²,0–2 km,4810.81,1910,5307.32,129,496.51
1,Hasta 60 m²,2–4 km,4283.33,1761,3600.00,305,-683.33
2,Hasta 60 m²,4–6 km,2525.66,1222,2618.33,402,92.68
3,Hasta 60 m²,6–8 km,2072.08,715,2382.98,205,310.90
4,Hasta 60 m²,Más de 8 km,2793.62,422,2511.11,98,-282.51
5,Más de 60–90 m²,0–2 km,4430.95,1062,4815.34,231,384.39
6,Más de 60–90 m²,2–4 km,3533.25,1690,2585.71,894,-947.54
7,Más de 60–90 m²,4–6 km,2120.00,1996,2079.47,1433,-40.53
8,Más de 60–90 m²,6–8 km,2061.54,1050,2000.00,795,-61.54
9,Más de 60–90 m²,Más de 8 km,2414.29,715,2069.05,523,-345.24


### H4. Terraza y precio por m²

En los 20 grupos de superficie y distancia con suficiente cobertura,
solo cinco presentan mayor mediana de precio por m² con terraza.
Las diferencias oscilan entre −947,54 y +496,51 €/m².

H4 queda refutada como tendencia general en esta comparación
descriptiva. No se aplicará una prima automática por terraza.
El resultado no demuestra un efecto causal negativo: persisten
diferencias entre viviendas que esta agrupación no controla.
HASTERRACE se mantendrá como variable candidata para el modelo.

### Resumen de hipótesis

| Hipótesis | Estado en entrenamiento | Evidencia y decisión |
|---|---|---|
| H1: mayor distancia, menor precio por m² | Apoyada descriptivamente | Spearman −0,532; asociación negativa en los cuatro grupos de superficie. Mantener distancia como variable candidata. |
| H2: mayor superficie, menor precio por m² | Refutada como tendencia general | Spearman +0,142; las medianas no disminuyen progresivamente. Conservar superficie sin imponer una relación negativa. |
| H3: ascensor asociado a mayor precio, especialmente en plantas altas | Apoyo parcial | Diferencia positiva en 45 de 46 grupos comparables; no aumenta continuamente con la planta. Mantener ascensor y planta, sin prima fija. |
| H4: terraza asociada a mayor precio por m² | Refutada como tendencia general | Diferencia positiva en 5 de 20 grupos comparables. No aplicar una prima automática. |

Los resultados son descriptivos y se limitan al entrenamiento.
Las agrupaciones no controlan todas las diferencias entre viviendas
y los anuncios repetidos pueden influir en las medianas. El mínimo
de 30 identificadores por categoría es un criterio de cobertura,
no una garantía de significación estadística.

### Qué significa para el responsable de valoraciones

La distancia y el ascensor ayudan a orientar la selección de
comparables. No resulta justificado aplicar reglas generales
de descuento por superficie o de incremento por terraza.
Las características deben valorarse conjuntamente y con revisión humana.

## 4. Variables del modelo

Se seleccionan características físicas y geográficas disponibles
al revisar un anuncio. En este ejercicio se supone que los datos
catastrales pueden consultarse en ese momento; la fuente no permite
verificar sus fechas históricas de actualización.

| Grupo | Variables | Justificación |
|---|---|---|
| Tamaño y distribución | CONSTRUCTEDAREA, ROOMNUMBER, BATHNUMBER | Describen la superficie y distribución de la vivienda. |
| Equipamiento | HASLIFT, HASTERRACE, HASAIRCONDITIONING, HASPARKINGSPACE | Permiten comparar características del anuncio. |
| Planta y antigüedad | FLOORCLEAN, CADCONSTRUCTIONYEAR | Pueden aportar información junto con ascensor, tamaño y localización. |
| Localización | DISTANCE_TO_CITY_CENTER, DISTANCE_TO_METRO, LONGITUDE, LATITUDE | Describen accesibilidad y posición aproximada. |
| Categorías | FLATLOCATIONID, AMENITYID, CADASTRALQUALITYID | Representan orientación interior/exterior, equipamiento y calidad catastral. Se tratarán como categorías, sin asumir distancias numéricas entre códigos. |

### Comprobación de fuga de información

Se excluyen UNITPRICE de las entradas, porque es el objetivo, y
PRICE, porque permite reconstruirlo junto con la superficie.
ASSETID se utiliza para controlar repeticiones, no para predecir.
PERIOD se conserva para la separación temporal.

No se utiliza CONSTRUCTIONYEAR por sus ausencias e incoherencias;
se selecciona el año catastral. Se excluyen PARKINGSPACEPRICE,
por las dudas de interpretación, e ISPARKINGSPACEINCLUDEDINPRICE,
porque duplica HASPARKINGSPACE en esta versión.

Los valores ausentes se tratarán dentro de la pipeline, aprendiendo
las medianas y categorías únicamente con entrenamiento. No se
utilizarán las marcas de auditoría como predictores.

### Qué significa para el responsable de valoraciones

El modelo utilizará características de la vivienda para ofrecer
una referencia. No recibirá el precio anunciado total con el que
podría reconstruir directamente la respuesta.

In [42]:
variables_numericas = [
    "CONSTRUCTEDAREA",
    "ROOMNUMBER",
    "BATHNUMBER",
    "FLOORCLEAN",
    "CADCONSTRUCTIONYEAR",
    "DISTANCE_TO_CITY_CENTER",
    "DISTANCE_TO_METRO",
    "LONGITUDE",
    "LATITUDE"
]

variables_binarias = [
    "HASLIFT",
    "HASTERRACE",
    "HASAIRCONDITIONING",
    "HASPARKINGSPACE"
]

variables_categoricas = [
    "FLATLOCATIONID",
    "AMENITYID",
    "CADASTRALQUALITYID"
]

variables_modelo = (
    variables_numericas
    + variables_binarias
    + variables_categoricas
)

# Comprobar que no hemos incluido la respuesta ni sus equivalentes
assert not {"UNITPRICE", "PRICE", "ASSETID"}.intersection(
    variables_modelo
)

# Entrenamiento: características y respuesta correcta
X_train = entrenamiento[variables_modelo].copy()
y_train = entrenamiento["UNITPRICE"].copy()

# Validación: para comparar modelos posteriormente
X_val = validacion[variables_modelo].copy()
y_val = validacion["UNITPRICE"].copy()

# No preparamos ni evaluamos aún el conjunto de prueba final
assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nValores ausentes en las variables de entrenamiento:")
nulos_train = X_train.isna().sum()
display(nulos_train[nulos_train > 0].to_frame("Ausentes"))

X_train: (34572, 16)
y_train: (34572,)
X_val: (15973, 16)
y_val: (15973,)

Valores ausentes en las variables de entrenamiento:


,Ausentes
ROOMNUMBER,2
FLOORCLEAN,1323
DISTANCE_TO_CITY_CENTER,1
DISTANCE_TO_METRO,1
LONGITUDE,1
LATITUDE,1
FLATLOCATIONID,2351


## 5. Separación temporal y referencias básicas

Se utilizan los dos primeros trimestres de 2018 para entrenar,
el tercero para validar y el cuarto para la evaluación final.
Se ha comprobado que ningún ASSETID aparece en dos conjuntos.

El cuarto trimestre apareció en la auditoría inicial, pero desde
la separación queda reservado para la evaluación final y no se
utiliza para seleccionar modelos ni ajustar parámetros.

### 5.1. Referencia sencilla: mediana del entrenamiento

Esta referencia predice el mismo precio por m² para todos los
anuncios: la mediana de UNITPRICE del entrenamiento. No utiliza
características de las viviendas.

Se elige la mediana porque minimiza la suma de errores absolutos
entre las predicciones constantes en entrenamiento. Se evaluará
en validación mediante MAE y RMSE, ambos expresados en €/m².

Esta referencia sirve como mínimo de comparación; no representa
por sí sola la regla de comparables propuesta para la agencia.

In [43]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Aprender la referencia SOLO del entrenamiento
mediana_train = float(y_train.median())

# Aplicar esa referencia a todas las viviendas de validación
pred_baseline_mediana = np.full(
    shape=len(y_val),
    fill_value=mediana_train
)

# Evaluar en validación
baseline_mae = mean_absolute_error(
    y_val, pred_baseline_mediana
)

baseline_rmse = np.sqrt(
    mean_squared_error(y_val, pred_baseline_mediana)
)

resultados_baselines = pd.DataFrame([{
    "Método": "Mediana del entrenamiento",
    "MAE (€/m²)": baseline_mae,
    "RMSE (€/m²)": baseline_rmse
}])

print(f"Precio predicho para todos los anuncios: {mediana_train:.2f} €/m²")
display(resultados_baselines.round(2))

Precio predicho para todos los anuncios: 3338.00 €/m²


,Método,MAE (€/m²),RMSE (€/m²)
0,Mediana del entrenamiento,1383.87,1734.5


### 5.2. Referencia por comparables

Cada anuncio de validación recibe la mediana del precio por m²
de los anuncios de entrenamiento de su mismo tramo de superficie
y distancia al centro.

Se utilizan los mismos intervalos de la exploración. Se exige
un mínimo de 30 identificadores distintos en el grupo de
entrenamiento. Si el grupo no existe, tiene cobertura insuficiente
o faltan datos para asignarlo, se utiliza la mediana global del
entrenamiento.

Esta regla representa una propuesta para la agencia ficticia,
no una práctica empresarial observada. Los comparables proceden
de trimestres anteriores para evitar utilizar precios del periodo
que estamos evaluando. Las medianas se calculan sobre registros,
por lo que las repeticiones pueden influir en ellas.

In [44]:
# Crear los mismos grupos para entrenamiento y validación
def crear_grupos_comparables(tabla):
    grupos = pd.DataFrame(index=tabla.index)

    grupos["superficie"] = pd.cut(
        tabla["CONSTRUCTEDAREA"],
        bins=[0, 60, 90, 120, np.inf],
        labels=["Hasta 60", "60–90", "90–120", "Más de 120"],
        include_lowest=True
    )

    grupos["distancia"] = pd.cut(
        tabla["DISTANCE_TO_CITY_CENTER"],
        bins=[0, 2, 4, 6, 8, np.inf],
        labels=["0–2", "2–4", "4–6", "6–8", "Más de 8"],
        include_lowest=True
    )

    return grupos


# Aprender las medianas SOLO con entrenamiento
grupos_train = crear_grupos_comparables(entrenamiento)
grupos_train["UNITPRICE"] = y_train
grupos_train["ASSETID"] = entrenamiento["ASSETID"]

referencias_comparables = (
    grupos_train.groupby(
        ["superficie", "distancia"], observed=True
    )
    .agg(
        mediana=("UNITPRICE", "median"),
        inmuebles=("ASSETID", "nunique")
    )
)

referencias_comparables = referencias_comparables[
    referencias_comparables["inmuebles"] >= 30
]

# Buscar una referencia para cada anuncio de validación
grupos_val = crear_grupos_comparables(validacion)

consulta_val = grupos_val.join(
    referencias_comparables[["mediana"]],
    on=["superficie", "distancia"],
    how="left",
    validate="many_to_one"
)

# Verificar que se mantiene el orden de las respuestas reales
assert consulta_val.index.equals(y_val.index)

sin_referencia = consulta_val["mediana"].isna()

pred_baseline_comparables = (
    consulta_val["mediana"]
    .fillna(mediana_train)
    .to_numpy()
)

mae_comparables = mean_absolute_error(
    y_val, pred_baseline_comparables
)

rmse_comparables = np.sqrt(
    mean_squared_error(y_val, pred_baseline_comparables)
)

# Reconstruir la tabla evita duplicar filas al repetir la celda
resultados_baselines = pd.DataFrame([
    {
        "Método": "Mediana del entrenamiento",
        "MAE (€/m²)": baseline_mae,
        "RMSE (€/m²)": baseline_rmse
    },
    {
        "Método": "Mediana por comparables",
        "MAE (€/m²)": mae_comparables,
        "RMSE (€/m²)": rmse_comparables
    }
])

mejora_comparables = (
    100 * (baseline_mae - mae_comparables) / baseline_mae
)

display(resultados_baselines.round(2))

print("Grupos de referencia disponibles:", len(referencias_comparables))
print("Anuncios que usan la mediana global:", int(sin_referencia.sum()))
print(f"Reducción del MAE frente a la referencia sencilla: {mejora_comparables:.2f}%")

,Método,MAE (€/m²),RMSE (€/m²)
0,Mediana del entrenamiento,1383.87,1734.50
1,Mediana por comparables,1070.74,1431.17


Grupos de referencia disponibles: 20
Anuncios que usan la mediana global: 0
Reducción del MAE frente a la referencia sencilla: 22.63%


### Resultados de las referencias básicas

La mediana global obtiene un MAE de 1.383,87 €/m² y un RMSE
de 1.734,50 €/m² en validación. La regla por comparables reduce
estos errores a 1.070,74 y 1.431,17 €/m², respectivamente.

La reducción del MAE es del 22,63 %. Hay 20 grupos de referencia
disponibles y ningún anuncio de validación necesita recurrir
a la mediana global.

### Qué significa para el responsable de valoraciones

Comparar viviendas por superficie y distancia ofrece una referencia
más precisa que utilizar un único precio para todas. Los modelos
deberán justificar su complejidad mejorando esta regla de comparables.
La reducción del error todavía no demuestra un ahorro económico.

## 6. Modelos candidatos

La implementación utiliza las pipelines y los estimadores
documentados por Scikit-learn developers (s. f.-a, s. f.-b).

### 6.1. Modelo sencillo: regresión Ridge

Ridge combina las características mediante una relación lineal
y penaliza coeficientes excesivamente grandes. Sirve como modelo
sencillo frente a las referencias básicas.

La preparación y el modelo se integran en una pipeline:
las variables numéricas se rellenan con la mediana y se
estandarizan; las binarias se rellenan con el valor más frecuente;
las categorías ausentes reciben un código específico y se
codifican mediante one-hot encoding.

Todos los valores de imputación, escalas y categorías se aprenden
solo con entrenamiento. Ridge se evalúa con alpha = 1 como modelo
sencillo de comparación. El ajuste mediante validación temporal
se realiza sobre Random Forest, seleccionado por su mejor
rendimiento en validación.

In [45]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge

# Preparación de cada tipo de variable
preparacion_numericas = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler())
])

preparacion_binarias = Pipeline([
    ("imputar", SimpleImputer(strategy="most_frequent"))
])

preparacion_categoricas = Pipeline([
    ("imputar", SimpleImputer(
        strategy="constant",
        fill_value=-999
    )),
    ("codificar", OneHotEncoder(handle_unknown="ignore"))
])

preprocesador = ColumnTransformer([
    ("numericas", preparacion_numericas, variables_numericas),
    ("binarias", preparacion_binarias, variables_binarias),
    ("categoricas", preparacion_categoricas, variables_categoricas)
])

# Pipeline completa: preparación + modelo
modelo_ridge = Pipeline([
    ("preparacion", preprocesador),
    ("modelo", Ridge(alpha=1.0, solver="lsqr"))
])

# Aprender únicamente del entrenamiento
modelo_ridge.fit(X_train, y_train)

# Evaluar en validación
pred_ridge = modelo_ridge.predict(X_val)

mae_ridge = mean_absolute_error(y_val, pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_val, pred_ridge))

resultados_modelos = pd.concat([
    resultados_baselines,
    pd.DataFrame([{
        "Método": "Regresión Ridge",
        "MAE (€/m²)": mae_ridge,
        "RMSE (€/m²)": rmse_ridge
    }])
], ignore_index=True)

display(resultados_modelos.round(2))

mejora_ridge = 100 * (mae_comparables - mae_ridge) / mae_comparables

print(f"Reducción del MAE frente a comparables: {mejora_ridge:.2f}%")
print("Predicciones negativas:", int((pred_ridge < 0).sum()))

,Método,MAE (€/m²),RMSE (€/m²)
0,Mediana del entrenamiento,1383.87,1734.50
1,Mediana por comparables,1070.74,1431.17
2,Regresión Ridge,789.37,1068.03


Reducción del MAE frente a comparables: 26.28%
Predicciones negativas: 34


Ridge obtiene un MAE de 789,37 €/m² y un RMSE de 1.068,03 €/m²
en validación. Reduce el MAE un 26,28 % respecto a la regla de
comparables.

Sin embargo, produce 34 predicciones negativas. Esto revela una
limitación de la regresión lineal sin restricciones para este
problema. Las métricas se presentan sin recortar esas predicciones.
El modelo no debería utilizarse directamente para recomendar precios.

### 6.2. Modelo de conjunto: Random Forest

Random Forest combina árboles de decisión para representar
relaciones no lineales e interacciones entre características.

Se limitan inicialmente la profundidad y el tamaño mínimo de las
hojas para controlar la complejidad. Estos valores son una
configuración inicial, no el resultado de un ajuste.

Se utiliza una copia independiente del preprocesador de Ridge.
Se conserva el escalado para compartir la preparación, aunque
los árboles no lo necesitan. La semilla queda fijada para
reproducir el entrenamiento.

In [46]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from time import perf_counter

modelo_rf = Pipeline([
    ("preparacion", clone(preprocesador)),
    ("modelo", RandomForestRegressor(
        n_estimators=100,
        max_depth=16,
        min_samples_leaf=5,
        max_features=0.8,
        random_state=SEED,
        n_jobs=-1
    ))
])

inicio = perf_counter()
modelo_rf.fit(X_train, y_train)
tiempo_rf = perf_counter() - inicio

pred_rf = modelo_rf.predict(X_val)

mae_rf = mean_absolute_error(y_val, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_val, pred_rf))

# Tabla completa; no duplica filas si se vuelve a ejecutar
resultados_modelos = pd.concat([
    resultados_baselines,
    pd.DataFrame([
        {
            "Método": "Regresión Ridge",
            "MAE (€/m²)": mae_ridge,
            "RMSE (€/m²)": rmse_ridge
        },
        {
            "Método": "Random Forest",
            "MAE (€/m²)": mae_rf,
            "RMSE (€/m²)": rmse_rf
        }
    ])
], ignore_index=True)

display(resultados_modelos.round(2))

print(f"Tiempo de entrenamiento: {tiempo_rf:.1f} segundos")
print(
    "Reducción del MAE frente a comparables:",
    f"{100 * (mae_comparables - mae_rf) / mae_comparables:.2f}%"
)
print(
    "Reducción del MAE frente a Ridge:",
    f"{100 * (mae_ridge - mae_rf) / mae_ridge:.2f}%"
)
print("Predicciones negativas:", int((pred_rf < 0).sum()))

,Método,MAE (€/m²),RMSE (€/m²)
0,Mediana del entrenamiento,1383.87,1734.50
1,Mediana por comparables,1070.74,1431.17
2,Regresión Ridge,789.37,1068.03
3,Random Forest,513.87,760.60


Tiempo de entrenamiento: 35.4 segundos
Reducción del MAE frente a comparables: 52.01%
Reducción del MAE frente a Ridge: 34.90%
Predicciones negativas: 0


Random Forest obtiene un MAE de 513,87 €/m² y un RMSE de
760,60 €/m² en validación. Reduce el MAE un 52,01 % frente
a comparables y un 34,90 % frente a Ridge. El entrenamiento
tardó 25,4 segundos y no produjo predicciones negativas.

### Qué significa para el responsable de valoraciones

Random Forest ofrece la referencia más precisa de los métodos
comparados en el tercer trimestre. Se selecciona como candidato
para validación temporal y ajuste. Estos resultados todavía
no corresponden a la prueba final ni garantizan el rendimiento futuro.

## 7. Métrica y coste de los errores

El MAE es la métrica principal porque expresa la desviación
absoluta media en €/m², una unidad directamente interpretable
por el responsable. Trata por igual las sobreestimaciones y
las subestimaciones, al no disponer de costes económicos
verificados que justifiquen ponderarlas de forma diferente.

El RMSE complementa al MAE al dar más peso a los errores grandes.
En Random Forest es 760,60 €/m², frente a un MAE de 513,87 €/m²,
lo que indica que algunos errores son considerablemente mayores.

Una sobreestimación podría hacer parecer razonable un anuncio
caro; una subestimación podría provocar una revisión innecesaria
o una recomendación demasiado baja. Estas consecuencias son
riesgos posibles, no pérdidas económicas medidas.

In [47]:
# Diagnóstico: el error de entrenamiento no mide generalización
pred_rf_train = modelo_rf.predict(X_train)
mae_rf_train = mean_absolute_error(y_train, pred_rf_train)

print(f"MAE de entrenamiento: {mae_rf_train:.2f} €/m²")
print(f"MAE de validación: {mae_rf:.2f} €/m²")

# Errores de validación
errores_val = validacion[["ASSETID", "UNITPRICE"]].copy()
errores_val["prediccion"] = pred_rf

# Positivo = sobreestimación; negativo = subestimación
errores_val["error_firmado"] = (
    errores_val["prediccion"] - errores_val["UNITPRICE"]
)
errores_val["error_absoluto"] = errores_val["error_firmado"].abs()

print(
    "Error firmado medio:",
    f"{errores_val['error_firmado'].mean():.2f} €/m²"
)
print(
    "Percentil 95 del error absoluto:",
    f"{errores_val['error_absoluto'].quantile(0.95):.2f} €/m²"
)

# Umbrales definidos SOLO con entrenamiento
q25_train = y_train.quantile(0.25)
q75_train = y_train.quantile(0.75)

errores_val["segmento_precio"] = pd.cut(
    errores_val["UNITPRICE"],
    bins=[-np.inf, q25_train, q75_train, np.inf],
    labels=["Bajo", "Intermedio", "Alto"]
)

resumen_errores = (
    errores_val.groupby("segmento_precio", observed=True)
    .agg(
        registros=("error_absoluto", "size"),
        inmuebles_distintos=("ASSETID", "nunique"),
        MAE=("error_absoluto", "mean"),
        error_firmado_medio=("error_firmado", "mean"),
        RMSE=("error_firmado", lambda s: np.sqrt((s ** 2).mean()))
    )
)

print(f"\nLímites de los segmentos: {q25_train:.2f} y {q75_train:.2f} €/m²")
display(resumen_errores.round(2))

MAE de entrenamiento: 330.50 €/m²
MAE de validación: 513.87 €/m²
Error firmado medio: -106.27 €/m²
Percentil 95 del error absoluto: 1598.05 €/m²

Límites de los segmentos: 2122.81 y 4610.22 €/m²


,registros,inmuebles_distintos,MAE,error_firmado_medio,RMSE
segmento_precio,,,,,
Bajo,3591,3253,298.24,175.60,493.75
Intermedio,8077,7219,443.35,41.64,607.77
Alto,4305,3901,826.07,-618.89,1118.07


### Diagnóstico de errores

El MAE de entrenamiento es 330,50 €/m², frente a 513,87 €/m²
en validación. Esta diferencia requiere comprobar la estabilidad
temporal; no demuestra por sí sola una única causa.

El error firmado medio en validación es −106,27 €/m² y el
percentil 95 del error absoluto alcanza 1.598,05 €/m².

El segmento alto presenta el mayor MAE: 826,07 €/m², con una
subestimación media de 618,89 €/m². En el segmento bajo se observa
una sobreestimación media de 175,60 €/m².

### Qué significa para el responsable de valoraciones

El modelo necesita especial cautela en anuncios de precio alto:
podría señalar como excesivos precios que no sabe representar bien.
Sus predicciones son referencias sujetas a revisión, no valoraciones
definitivas ni intervalos garantizados.

## 8. Validación temporal y ajuste

Se ajusta Random Forest mediante una búsqueda de cuatro
configuraciones de profundidad y tamaño mínimo de hoja.

Se emplean dos divisiones temporales expansivas: primer trimestre
para entrenar y segundo para evaluar; después, primero y segundo
para entrenar y tercero para evaluar. No se dividen aleatoriamente
las filas ni se comparten identificadores entre entrenamiento
y evaluación en cada división.

El tercer trimestre, utilizado previamente para comparar candidatos,
se incorpora al ajuste. Por ello, sus resultados no constituyen
una evaluación final independiente. El cuarto trimestre permanece
reservado. Disponer de solo dos divisiones limita la evidencia
sobre estabilidad temporal.

La búsqueda y evaluación utilizan las herramientas descritas
por Scikit-learn developers (s. f.-b), con divisiones temporales
adaptadas a los trimestres disponibles.

In [48]:
from sklearn.model_selection import GridSearchCV

# Desarrollo: los tres primeros trimestres, sin tocar prueba
desarrollo = pd.concat(
    [entrenamiento, validacion]
).sort_values("PERIOD", kind="stable")

X_desarrollo = desarrollo[variables_modelo].copy()
y_desarrollo = desarrollo["UNITPRICE"].copy()

periodos_desarrollo = desarrollo["PERIOD"].to_numpy()

divisiones_temporales = [
    (
        np.flatnonzero(periodos_desarrollo == 201803),
        np.flatnonzero(periodos_desarrollo == 201806)
    ),
    (
        np.flatnonzero(
            np.isin(periodos_desarrollo, [201803, 201806])
        ),
        np.flatnonzero(periodos_desarrollo == 201809)
    )
]

# Verificar separación temporal y de identificadores
for indices_train, indices_eval in divisiones_temporales:
    bloque_train = desarrollo.iloc[indices_train]
    bloque_eval = desarrollo.iloc[indices_eval]

    assert bloque_train["PERIOD"].max() < bloque_eval["PERIOD"].min()
    assert set(bloque_train["ASSETID"]).isdisjoint(
        set(bloque_eval["ASSETID"])
    )

parametros = {
    "modelo__max_depth": [12, 16],
    "modelo__min_samples_leaf": [5, 10]
}

busqueda_rf = GridSearchCV(
    estimator=clone(modelo_rf),
    param_grid=parametros,
    scoring="neg_mean_absolute_error",
    cv=divisiones_temporales,
    refit=True,
    n_jobs=1,
    return_train_score=False,
    error_score="raise"
)

inicio_busqueda = perf_counter()
busqueda_rf.fit(X_desarrollo, y_desarrollo)
tiempo_busqueda = perf_counter() - inicio_busqueda

cv_resultados = pd.DataFrame(busqueda_rf.cv_results_)

tabla_cv = pd.DataFrame({
    "Profundidad": cv_resultados["param_modelo__max_depth"],
    "Mínimo por hoja": cv_resultados["param_modelo__min_samples_leaf"],
    "MAE división 1": -cv_resultados["split0_test_score"],
    "MAE división 2": -cv_resultados["split1_test_score"],
    "MAE medio": -cv_resultados["mean_test_score"],
    "Desviación entre divisiones": cv_resultados["std_test_score"]
}).sort_values("MAE medio")

display(tabla_cv.round(2))

print("Mejores parámetros:", busqueda_rf.best_params_)
print(f"MAE medio seleccionado: {-busqueda_rf.best_score_:.2f} €/m²")
print(f"Tiempo de búsqueda: {tiempo_busqueda:.1f} segundos")

# GridSearchCV ya lo reajusta con los tres primeros trimestres
modelo_final = busqueda_rf.best_estimator_

,Profundidad,Mínimo por hoja,MAE división 1,MAE división 2,MAE medio,Desviación entre divisiones
2,16,5,528.91,515.29,522.10,6.81
3,16,10,541.84,526.11,533.97,7.86
0,12,5,544.38,534.84,539.61,4.77
1,12,10,550.47,538.41,544.44,6.03


Mejores parámetros: {'modelo__max_depth': 16, 'modelo__min_samples_leaf': 5}
MAE medio seleccionado: 522.10 €/m²
Tiempo de búsqueda: 186.7 segundos


La mejor configuración entre las cuatro evaluadas utiliza profundidad
máxima 16 y un mínimo de 5 registros por hoja. Coincide con la
configuración inicial.

Obtiene un MAE medio de 522,10 €/m²: 528,91 en la primera división
temporal y 515,29 en la segunda. La desviación entre ambas es
6,81 €/m². Los resultados son cercanos, pero solo se dispone de
dos divisiones, por lo que la evidencia de estabilidad es limitada.

La búsqueda tardó 186,3 segundos. La pipeline seleccionada se
reajustó con los tres primeros trimestres. A partir de este punto
se fija la configuración antes de evaluar el cuarto trimestre.

### Evaluación final en el cuarto trimestre

Se evalúa la configuración seleccionada sobre el cuarto trimestre.
Las dos referencias se recalculan utilizando únicamente los tres
primeros trimestres, para comparar métodos con el mismo periodo
de información disponible.

Los resultados de prueba no se utilizarán para ajustar el modelo.
Las métricas se calculan por registro; los inmuebles con varios
anuncios pueden tener más peso, una limitación que se documentará.

In [49]:
# Entradas de prueba: se utilizan una vez fijado el modelo
X_test = prueba[variables_modelo].copy()
y_test = prueba["UNITPRICE"].copy()

assert set(desarrollo["ASSETID"]).isdisjoint(set(prueba["ASSETID"]))

# 1. Referencia global aprendida de los tres primeros trimestres
mediana_desarrollo = float(y_desarrollo.median())
pred_test_mediana = np.full(len(y_test), mediana_desarrollo)

# 2. Referencia por comparables, también aprendida de desarrollo
grupos_desarrollo = crear_grupos_comparables(desarrollo)
grupos_desarrollo["UNITPRICE"] = y_desarrollo
grupos_desarrollo["ASSETID"] = desarrollo["ASSETID"]

referencias_finales = (
    grupos_desarrollo.groupby(
        ["superficie", "distancia"], observed=True
    )
    .agg(
        mediana=("UNITPRICE", "median"),
        inmuebles=("ASSETID", "nunique")
    )
)

referencias_finales = referencias_finales[
    referencias_finales["inmuebles"] >= 30
]

consulta_test = crear_grupos_comparables(prueba).join(
    referencias_finales[["mediana"]],
    on=["superficie", "distancia"],
    how="left",
    validate="many_to_one"
)

assert consulta_test.index.equals(y_test.index)

pred_test_comparables = (
    consulta_test["mediana"]
    .fillna(mediana_desarrollo)
    .to_numpy()
)

# 3. Modelo final: ya entrenado, no volvemos a ajustarlo
pred_test_rf = modelo_final.predict(X_test)

# Guardar resultados para reutilizarlos en los análisis posteriores
predicciones_test = {
    "Mediana de desarrollo": pred_test_mediana,
    "Mediana por comparables": pred_test_comparables,
    "Random Forest final": pred_test_rf
}

resultados_test = pd.DataFrame([
    {
        "Método": nombre,
        "MAE (€/m²)": mean_absolute_error(y_test, pred),
        "RMSE (€/m²)": np.sqrt(mean_squared_error(y_test, pred))
    }
    for nombre, pred in predicciones_test.items()
])

display(resultados_test.round(2))

mae_test_comparables = mean_absolute_error(
    y_test, pred_test_comparables
)
mae_test_rf = mean_absolute_error(y_test, pred_test_rf)

mejora_test = (
    100 * (mae_test_comparables - mae_test_rf)
    / mae_test_comparables
)

print(f"Reducción del MAE frente a comparables: {mejora_test:.2f}%")
print(
    "Anuncios que usan la mediana global:",
    int(consulta_test["mediana"].isna().sum())
)
print("Predicciones negativas:", int((pred_test_rf < 0).sum()))

,Método,MAE (€/m²),RMSE (€/m²)
0,Mediana de desarrollo,1389.61,1746.05
1,Mediana por comparables,1077.08,1444.92
2,Random Forest final,516.87,752.80


Reducción del MAE frente a comparables: 52.01%
Anuncios que usan la mediana global: 0
Predicciones negativas: 0


En la prueba final, Random Forest obtiene un MAE de 516,87 €/m²
y un RMSE de 752,80 €/m². La referencia por comparables obtiene
1.077,08 y 1.444,92 €/m², respectivamente.

El modelo reduce el MAE un 52,01 % frente a comparables.
No genera predicciones negativas y todos los anuncios encuentran
un grupo de referencia con cobertura suficiente.

### Qué significa para el responsable de valoraciones

El modelo mejora la precisión de la regla propuesta de comparables
en el trimestre reservado. Esto respalda su uso como referencia
para revisión humana, pero no demuestra ahorro económico ni
garantiza resultados equivalentes en otros años o mercados.

## 9. Interpretación

### 9.1. Importancia de las variables por permutación

Se estima la importancia sobre una muestra reproducible de
2.000 registros de prueba, con tres repeticiones por variable.
Se limita la muestra para controlar el tiempo de ejecución.

La importancia representa el aumento del MAE al desordenar una
variable. No mide causalidad ni indica si su efecto es positivo
o negativo. Las variables relacionadas, como coordenadas y
distancia al centro, pueden compartir información y reducir
sus importancias individuales.

Este análisis se realiza después de fijar y evaluar el modelo.
No se utilizará para volver a seleccionar variables ni ajustar
parámetros con el conjunto de prueba.

In [50]:
from sklearn.inspection import permutation_importance

# Muestra reproducible para interpretar, sin volver a entrenar
X_interpretacion = X_test.sample(
    n=min(2000, len(X_test)),
    random_state=SEED
)
y_interpretacion = y_test.loc[X_interpretacion.index]

inicio_importancia = perf_counter()

resultado_importancia = permutation_importance(
    modelo_final,
    X_interpretacion,
    y_interpretacion,
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    random_state=SEED,
    n_jobs=1
)

tabla_importancias = pd.DataFrame({
    "Variable": X_interpretacion.columns,
    "Aumento del MAE (€/m²)": resultado_importancia.importances_mean,
    "Desviación entre repeticiones": resultado_importancia.importances_std
}).sort_values("Aumento del MAE (€/m²)", ascending=False)

display(tabla_importancias.round(2))

print(
    "Tiempo de interpretación:",
    f"{perf_counter() - inicio_importancia:.1f} segundos"
)

,Variable,Aumento del MAE (€/m²),Desviación entre repeticiones
5,DISTANCE_TO_CITY_CENTER,582.22,11.00
8,LATITUDE,550.33,19.33
7,LONGITUDE,276.57,3.90
0,CONSTRUCTEDAREA,111.27,4.00
9,HASLIFT,59.70,1.30
3,FLOORCLEAN,55.24,4.54
4,CADCONSTRUCTIONYEAR,33.17,1.68
12,HASPARKINGSPACE,27.60,0.51
1,ROOMNUMBER,20.33,0.56
11,HASAIRCONDITIONING,14.33,2.03


Tiempo de interpretación: 3.5 segundos


La distancia al centro presenta el mayor aumento del MAE al
permutarla: 582,22 €/m², seguida de latitud (550,33) y longitud
(276,57). La superficie aporta 111,27 €/m²; ascensor y planta,
59,70 y 55,24, respectivamente.

Estos resultados son coherentes con H1 y con el papel de ascensor
y planta observado en H3. La importancia de superficie no implica
que su relación con el precio sea negativa, como proponía H2.

Terraza y equipamiento presentan escasa importancia individual
en esta muestra. Las importancias no son aditivas ni causales,
y pueden verse afectadas por la información compartida entre variables.

In [51]:
# Elegir tres casos de distintos niveles de precio
indices_casos = [
    (y_test - y_test.quantile(q)).abs().idxmin()
    for q in [0.10, 0.50, 0.90]
]

predicciones_guardadas = pd.Series(
    pred_test_rf, index=X_test.index
)

casos = X_test.loc[
    indices_casos,
    ["CONSTRUCTEDAREA", "DISTANCE_TO_CITY_CENTER",
     "HASLIFT", "FLOORCLEAN", "HASTERRACE"]
].copy()

casos.insert(0, "Caso", ["Precio bajo", "Precio mediano", "Precio alto"])
casos["Precio real (€/m²)"] = y_test.loc[indices_casos]
casos["Predicción (€/m²)"] = predicciones_guardadas.loc[indices_casos]
casos["Error firmado (€/m²)"] = (
    casos["Predicción (€/m²)"] - casos["Precio real (€/m²)"]
)

display(casos.round(2))

# Sensibilidad local: cambiar una característica cada vez.
# Son escenarios artificiales, no efectos causales.
sensibilidad = []

for indice, nombre in zip(
    indices_casos, ["Precio bajo", "Precio mediano", "Precio alto"]
):
    original = X_test.loc[[indice]].copy()
    pred_original = float(predicciones_guardadas.loc[indice])

    escenarios = {
        "Superficie −10 %": ("CONSTRUCTEDAREA", 0.9),
        "Superficie +10 %": ("CONSTRUCTEDAREA", 1.1)
    }

    for escenario, (columna, factor) in escenarios.items():
        alternativa = original.copy()
        alternativa[columna] = original[columna] * factor

        pred_alternativa = float(modelo_final.predict(alternativa)[0])

        sensibilidad.append({
            "Caso": nombre,
            "Cambio": escenario,
            "Cambio en predicción (€/m²)": pred_alternativa - pred_original
        })

    if original["HASLIFT"].iloc[0] in [0, 1]:
        alternativa = original.copy()
        nuevo_ascensor = 1 - int(original["HASLIFT"].iloc[0])
        alternativa["HASLIFT"] = nuevo_ascensor

        sensibilidad.append({
            "Caso": nombre,
            "Cambio": f"Ascensor pasa a {nuevo_ascensor}",
            "Cambio en predicción (€/m²)": (
                float(modelo_final.predict(alternativa)[0]) - pred_original
            )
        })

display(pd.DataFrame(sensibilidad).round(2))

,Caso,CONSTRUCTEDAREA,DISTANCE_TO_CITY_CENTER,HASLIFT,FLOORCLEAN,HASTERRACE,Precio real (€/m²),Predicción (€/m²),Error firmado (€/m²)
23082,Precio bajo,90,4.44,0,2.0,1,1766.67,1642.85,-123.82
80414,Precio mediano,135,5.74,1,3.0,0,3585.19,2937.77,-647.42
34346,Precio alto,72,0.56,0,4.0,0,6041.67,4251.30,-1790.36


,Caso,Cambio,Cambio en predicción (€/m²)
0,Precio bajo,Superficie −10 %,57.83
1,Precio bajo,Superficie +10 %,-31.97
2,Precio bajo,Ascensor pasa a 1,312.25
3,Precio mediano,Superficie −10 %,45.46
4,Precio mediano,Superficie +10 %,-66.17
5,Precio mediano,Ascensor pasa a 0,-683.29
6,Precio alto,Superficie −10 %,-1.24
7,Precio alto,Superficie +10 %,-50.82
8,Precio alto,Ascensor pasa a 1,1415.46


### 9.2. Tres predicciones individuales

Se seleccionan anuncios próximos a los percentiles 10, 50 y 90
del precio real de prueba, sin seleccionar por precisión.

El caso de precio bajo presenta un error de −123,82 €/m².
El mediano se subestima en 647,42 €/m² y el alto en
1.790,36 €/m². Este último es una vivienda de 72 m², situada
a 0,56 km del centro, en planta 4 y sin ascensor.

El análisis de sensibilidad muestra que cambiar ascensor de
ausente a presente aumenta la predicción del caso alto en
1.415,46 €/m². En el caso mediano, retirarlo reduce la predicción
en 683,29 €/m². Los cambios de superficie tienen efectos menores
en los escenarios examinados.

Estas modificaciones describen respuestas locales del modelo,
no efectos causales ni valoraciones de reformas. Pueden crear
combinaciones poco representadas en los datos y no explican
por completo cada predicción.

### Qué significa para el responsable de valoraciones

Una buena precisión media puede ocultar errores importantes en
anuncios individuales. Los casos con grandes discrepancias deben
revisarse con información adicional, sin asumir que el anuncio
está mal valorado.

## 10. Equidad, ética y riesgos

Se comparan los errores por distancia al centro y el efecto de
dar el mismo peso a cada identificador. Se reutilizan las
predicciones finales, sin volver a entrenar ni ajustar el modelo.

In [52]:
# Reutilizamos las predicciones finales, sin entrenar ni ajustar
auditoria_test = prueba[
    ["ASSETID", "DISTANCE_TO_CITY_CENTER", "UNITPRICE"]
].copy()

auditoria_test["prediccion"] = pred_test_rf
auditoria_test["error_firmado"] = (
    auditoria_test["prediccion"] - auditoria_test["UNITPRICE"]
)
auditoria_test["error_absoluto"] = (
    auditoria_test["error_firmado"].abs()
)

auditoria_test["zona_distancia"] = pd.cut(
    auditoria_test["DISTANCE_TO_CITY_CENTER"],
    bins=[0, 2, 4, 6, 8, np.inf],
    labels=["0–2 km", "2–4 km", "4–6 km", "6–8 km", "Más de 8 km"],
    include_lowest=True
)

errores_por_zona = (
    auditoria_test.groupby("zona_distancia", observed=True)
    .agg(
        registros=("error_absoluto", "size"),
        inmuebles_distintos=("ASSETID", "nunique"),
        MAE=("error_absoluto", "mean"),
        error_firmado_medio=("error_firmado", "mean"),
        RMSE=("error_firmado", lambda s: np.sqrt((s ** 2).mean()))
    )
)

display(errores_por_zona.round(2))

print(
    "Registros sin grupo de distancia:",
    int(auditoria_test["zona_distancia"].isna().sum())
)

# Primero promediamos los errores de cada identificador.
# Después damos el mismo peso a todos los identificadores.
mae_por_identificador = (
    auditoria_test.groupby("ASSETID")["error_absoluto"]
    .mean()
    .mean()
)

print(f"MAE por registro: {mae_test_rf:.2f} €/m²")
print(
    "MAE con igual peso por identificador:",
    f"{mae_por_identificador:.2f} €/m²"
)

,registros,inmuebles_distintos,MAE,error_firmado_medio,RMSE
zona_distancia,,,,,
0–2 km,8036,5730,730.94,-100.61,1004.94
2–4 km,13288,9450,581.65,-158.19,817.33
4–6 km,11283,7993,417.54,-141.99,615.43
6–8 km,6666,4836,387.16,-125.03,566.18
Más de 8 km,4997,3606,397.68,-115.09,580.73


Registros sin grupo de distancia: 0
MAE por registro: 516.87 €/m²
MAE con igual peso por identificador: 524.15 €/m²


El MAE es mayor cerca del centro: 730,94 €/m² en 0–2 km,
frente a 387,16 €/m² en 6–8 km. Todos los grupos presentan
subestimación media. Estas diferencias no demuestran por sí
solas discriminación; pueden influir los precios y la diversidad
de viviendas. No disponemos de atributos demográficos para
evaluar equidad entre grupos protegidos.

Al dar el mismo peso a cada identificador, el MAE pasa de
516,87 a 524,15 €/m² (+1,41 %). La diferencia es pequeña,
pero no elimina la influencia de las repeticiones al entrenar.

| Riesgo | Medida propuesta |
|---|---|
| Mayor error en zonas centrales y anuncios caros | Revisar discrepancias importantes y mostrar métricas por segmento. |
| Datos históricos y cobertura parcial | Exigir datos recientes y nueva validación antes de un uso actual. |
| Repeticiones e información incompleta | Mantener controles por identificador y registrar ausencias. |
| Identificación mediante localización | No reidentificar personas ni tratar coordenadas perturbadas como direcciones exactas; utilizar mapas agregados. |
| Confundir la predicción con una tasación | Presentarla como referencia de precios anunciados, sujeta a revisión humana. |

### Qué significa para el responsable de valoraciones

La precisión no es uniforme. Se requiere especial revisión
en zonas centrales y deben evitarse decisiones automáticas
que puedan perjudicar a propietarios o compradores.

## 11. Recomendación

Se recomienda Random Forest como prototipo de apoyo a la revisión
de anuncios históricos. En prueba obtiene un MAE de 516,87 €/m²
frente a 1.077,08 de la regla de comparables: una reducción del
52,01 %, equivalente a 560,21 €/m² menos de error absoluto medio.

Esta mejora es un indicador de precisión predictiva, no un ahorro
económico demostrado. El responsable podría priorizar revisiones
según la diferencia entre el precio anunciado y la referencia del
modelo, considerando cobertura y características del inmueble.

No se recomienda fijar precios automáticamente ni utilizar este
modelo para el mercado actual, otras ciudades o inmuebles fuera
del ámbito de los datos. Requieren especial revisión las zonas
centrales, los anuncios caros y los casos con información incompleta.

El bloque 4 incorporará la interfaz, validaciones de entrada,
registro de predicciones y seguimiento de errores, ausencias y
cambios en los datos. Un piloto deberá medir tiempo de revisión
y utilidad de las alertas antes de afirmar beneficios económicos.

### Qué significa para el responsable de valoraciones

El prototipo mejora la regla sencilla en el periodo evaluado.
Su siguiente paso es un uso supervisado y medido, manteniendo
la decisión final en manos del profesional.

## 12. Guardado y comprobación del modelo

Se guarda una única pipeline que incorpora las reglas fijas de
limpieza, la imputación, la codificación y el Random Forest
entrenado. Recibe las 16 características seleccionadas y los
campos ASSETID y PERIOD, utilizados únicamente para identificar
la corrección geográfica conocida.

El modelo se recupera mediante joblib.load
(Joblib developers, s. f.).

La limpieza integrada reproduce los datos utilizados al entrenar.
Tras guardar y recargar la pipeline, una predicción de comprobación
se mantiene en 2.862,21 €/m². El archivo ocupa 18,09 MB.

El archivo madrid_preprocessing.py debe acompañar a model.joblib
y estar disponible para Python al cargarlo. Las versiones del
entorno quedan registradas en la salida del código.

### Qué significa para el responsable de valoraciones

El modelo puede recuperarse sin volver a entrenarlo y conserva
el tratamiento de las entradas. La comprobación confirma una
predicción en el entorno actual; el despliegue deberá verificar
también su funcionamiento en un entorno nuevo.

In [53]:
%%writefile madrid_preprocessing.py
import numpy as np

def limpiar_entradas(tabla):
    """Aplicar las reglas fijas del proyecto sin aprender de los datos."""
    datos = tabla.copy()

    # No se puede predecir sin una superficie válida
    superficie = datos["CONSTRUCTEDAREA"]
    if (superficie.isna() | ~np.isfinite(superficie) | (superficie <= 0)).any():
        raise ValueError("La superficie debe ser un número positivo y finito.")

    # Dormitorios extremadamente poco plausibles
    dormitorios = datos["ROOMNUMBER"]
    ratio = superficie / dormitorios.where(dormitorios > 0)
    sospechosos = (dormitorios > 0) & (ratio < 5)

    datos.loc[sospechosos, "ROOMNUMBER"] = np.nan

    # Corrección del registro geográfico auditado.
    # ASSETID y PERIOD sirven para esta regla, no como predictores.
    caso_geografico = (
        datos["ASSETID"].eq("A7501251371917084834")
        & datos["PERIOD"].eq(201806)
    )

    columnas_geograficas = [
        "LONGITUDE",
        "LATITUDE",
        "DISTANCE_TO_CITY_CENTER",
        "DISTANCE_TO_METRO"
    ]
    datos.loc[caso_geografico, columnas_geograficas] = np.nan

    return datos

Overwriting madrid_preprocessing.py


In [54]:
from pathlib import Path
import shutil
import joblib
import sklearn

from sklearn.preprocessing import FunctionTransformer
from madrid_preprocessing import limpiar_entradas

# Las 16 características y los dos campos para la regla de limpieza
columnas_entrada = ["ASSETID", "PERIOD"] + variables_modelo

entradas_originales = datos_originales.loc[
    X_desarrollo.index, columnas_entrada
].copy()

limpieza = FunctionTransformer(
    limpiar_entradas,
    validate=False
)

# FunctionTransformer no aprende estadísticas de los datos
limpieza.fit(entradas_originales)

# Comprobar que reproduce exactamente la limpieza usada al entrenar
entradas_comprobadas = limpieza.transform(entradas_originales)

pd.testing.assert_frame_equal(
    entradas_comprobadas[variables_modelo],
    X_desarrollo,
    check_dtype=False
)

# Incorporar el modelo YA entrenado, sin volver a ajustarlo
pipeline_completa = Pipeline([
    ("limpieza", limpieza),
    ("prediccion", modelo_final)
])

# Guardar en project/; si el notebook ya está dentro, usar esa carpeta
carpeta_modelo = (
    Path(".") if Path.cwd().name == "project" else Path("project")
)
carpeta_modelo.mkdir(exist_ok=True)

ruta_modelo = carpeta_modelo / "model.joblib"
ruta_modulo = carpeta_modelo / "madrid_preprocessing.py"

if Path("madrid_preprocessing.py").resolve() != ruta_modulo.resolve():
    shutil.copy2("madrid_preprocessing.py", ruta_modulo)

joblib.dump(pipeline_completa, ruta_modelo, compress=3)

# Recargar y comprobar una predicción con un registro de desarrollo
modelo_recargado = joblib.load(ruta_modelo)

ejemplo_original = entradas_originales.iloc[[0]]
ejemplo_preparado = X_desarrollo.iloc[[0]]

pred_antes = modelo_final.predict(ejemplo_preparado)
pred_despues = modelo_recargado.predict(ejemplo_original)

np.testing.assert_allclose(
    pred_antes, pred_despues, rtol=1e-10, atol=1e-8
)

print("Limpieza comprobada: coincide con la utilizada al entrenar.")
print(f"Predicción antes de guardar: {pred_antes[0]:.2f} €/m²")
print(f"Predicción después de cargar: {pred_despues[0]:.2f} €/m²")
print("Modelo guardado en:", ruta_modelo)
print("Módulo de limpieza:", ruta_modulo)
print(f"Tamaño del modelo: {ruta_modelo.stat().st_size / 1_000_000:.2f} MB")
print("scikit-learn:", sklearn.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("joblib:", joblib.__version__)

Limpieza comprobada: coincide con la utilizada al entrenar.
Predicción antes de guardar: 2862.21 €/m²
Predicción después de cargar: 2862.21 €/m²
Modelo guardado en: project/model.joblib
Módulo de limpieza: project/madrid_preprocessing.py
Tamaño del modelo: 18.09 MB
scikit-learn: 1.6.1
pandas: 2.2.3
numpy: 2.4.6
joblib: 1.6.0


In [55]:
# Descarga opcional: no se activa al ejecutar todo el notebook
#Si se quiere descaragar cambiar False por True
DESCARGAR_ARCHIVOS = False

if DESCARGAR_ARCHIVOS:
    try:
        from google.colab import files
    except ImportError:
        print("Fuera de Colab, los archivos están en:", carpeta_modelo.resolve())
    else:
        files.download(str(ruta_modelo))
        files.download(str(ruta_modulo))
else:
    print("Archivos guardados en:", carpeta_modelo.resolve())
    print("Descarga automática desactivada.")

Archivos guardados en: /content/project
Descarga automática desactivada.


## 13. Referencias

Joblib developers. *joblib.load*. Joblib documentation.
https://joblib.readthedocs.io/en/stable/generated/joblib.load.html

Pandas development team.*pandas documentation*.
Recuperado el 16 de noviembre de 2026, de
https://pandas.pydata.org/docs/

Plotly Technologies Inc.*Plotly Python graphing library*.
Recuperado el 16 de noviembre de 2026, de
https://plotly.com/python/

Rey-Blanco, D., Arbues, P., Lopez, F., & Paez, A.
*idealista18* [Conjunto de datos y código]. GitHub.
Versión utilizada: commit e9ef80618f8a9cc28f672ac138f298bbd519918e.
https://github.com/paezha/idealista18/tree/e9ef80618f8a9cc28f672ac138f298bbd519918e

Scikit-learn developers.*RandomForestRegressor*.
Scikit-learn 1.6.1 documentation.
https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.RandomForestRegressor.html

Scikit-learn developers. *User guide*.
Scikit-learn 1.6.1 documentation.
https://scikit-learn.org/1.6/user_guide.html

In [56]:
tiempo_total = perf_counter() - inicio_notebook

print(f"Tiempo total: {tiempo_total:.1f} segundos")
print(f"Tiempo total: {tiempo_total / 60:.2f} minutos")
print("Cumple el límite de 10 minutos:", tiempo_total < 600)

Tiempo total: 237.1 segundos
Tiempo total: 3.95 minutos
Cumple el límite de 10 minutos: True
